# 13 · Where RL meets network optimization: Q-routing

**Goal:** connect the two tracks. Shortest path is a deterministic MDP: states are nodes, actions are outgoing arcs, reward is $-c_{ij}$, and $t$ is terminal. Its Bellman optimality equation

$$V^*(i)=\max_{(i,j)}\big[-c_{ij}+V^*(j)\big],\qquad V^*(t)=0$$

is exactly the shortest-path recursion $\mathrm{dist}(i,t)=\min_j[c_{ij}+\mathrm{dist}(j,t)]$. So $V^*(i)=-\mathrm{dist}(i,t)$.

This notebook learns routes with **Q-learning** from experience, checks them against Dijkstra, and then lets an adversary interdict the network so you can watch the agent **adapt** without being told what changed.

In [ ]:
import itertools
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
import heapq

def dijkstra(n, arcs, s):
    # arcs: list of (i, j, length) with length >= 0. Returns distances and predecessor array.
    adj = [[] for _ in range(n)]
    for i, j, c in arcs:
        adj[i].append((j, c))
    dist = np.full(n, np.inf); dist[s] = 0.0
    pred = [None] * n
    done = [False] * n
    pq = [(0.0, s)]
    while pq:
        d, u = heapq.heappop(pq)
        if done[u]:
            continue
        done[u] = True                       # u's label is now permanent
        for v, c in adj[u]:
            if d + c < dist[v]:
                dist[v] = d + c; pred[v] = u
                heapq.heappush(pq, (dist[v], v))
    return dist, pred

def path_to(pred, t):
    path = [t]
    while pred[path[-1]] is not None:
        path.append(pred[path[-1]])
    return path[::-1]

In [ ]:
# A small supply network: s = 0 (depot), t = 7 (destination).
# Each arc: (i, j, length c, extra delay d if interdicted, interdiction cost b)
ARCS = [
    (0, 1, 3, 8, 2), (0, 2, 4, 8, 2), (0, 3, 6, 8, 2),
    (1, 4, 4, 8, 1), (1, 2, 2, 8, 1), (2, 4, 3, 8, 1), (2, 5, 5, 8, 1),
    (3, 5, 3, 8, 1), (3, 6, 7, 8, 1), (4, 7, 6, 8, 1), (4, 5, 2, 8, 1),
    (5, 7, 4, 8, 1), (5, 6, 2, 8, 1), (6, 7, 3, 8, 2),
]
n, s, t = 8, 0, 7
m = len(ARCS)
c = np.array([a[2] for a in ARCS], float)
d = np.array([a[3] for a in ARCS], float)
b = np.array([a[4] for a in ARCS], float)
POS = {0: (0, 1), 1: (1, 2), 2: (1, 1), 3: (1, 0), 4: (2, 2), 5: (2, 1), 6: (2, 0), 7: (3, 1)}

In [ ]:
out_arcs = [[k for k, a in enumerate(ARCS) if a[0] == v] for v in range(n)]
assert all(out_arcs[v] for v in range(n) if v != t)          # no dead ends

def dist_to_t(lengths):
    rev = [(ARCS[k][1], ARCS[k][0], lengths[k]) for k in range(m)]
    dd, _ = dijkstra(n, rev, t)                               # Dijkstra on the reversed graph
    return dd

def q_routing(lengths, Q=None, episodes=3000, alpha=0.5, eps=0.3, seed=0):
    rng = np.random.default_rng(seed)
    Q = np.zeros(m) if Q is None else Q.copy()                # Q indexed by arc = (state, action)
    target_d = dist_to_t(lengths)
    starts = [v for v in range(n) if v != t]
    errs = []
    for ep in range(episodes):
        v = rng.choice(starts)
        for _ in range(50):
            acts = out_arcs[v]
            k = rng.choice(acts) if rng.random() < eps else acts[int(np.argmax(Q[acts]))]
            w = ARCS[k][1]
            v_next = 0.0 if w == t else Q[out_arcs[w]].max()
            Q[k] += alpha * (-lengths[k] + v_next - Q[k])
            v = w
            if v == t:
                break
        if ep % 25 == 0:
            V = np.array([Q[out_arcs[u]].max() for u in starts])
            errs.append(np.max(np.abs(V + target_d[starts])))
    return Q, errs

Q0, errs0 = q_routing(c)
V0 = np.array([Q0[out_arcs[u]].max() for u in range(n) if u != t])
print("learned  -V:", (-V0).round(3))
print("Dijkstra   :", dist_to_t(c)[:t])
assert errs0[-1] < 1e-3

## The adversary moves

Pick the worst-case 2-arc interdiction by brute force (notebook 12 does this with a MILP). Then *continue* training from the old $Q$ on the new lengths. The agent only sees new rewards; it is never told which arcs changed.

In [ ]:
def worst_case(B):
    best = (-np.inf, ())
    for r in range(m + 1):
        for S_ in itertools.combinations(range(m), r):
            if b[list(S_)].sum() <= B:
                L = c.copy(); L[list(S_)] += d[list(S_)]
                val = dist_to_t(L)[s]
                best = max(best, (val, S_))
    return best

val, hit = worst_case(2)
lengths1 = c.copy(); lengths1[list(hit)] += d[list(hit)]
print("adversary interdicts", [ARCS[k][:2] for k in hit], " -> new s-t distance", val)

Q1, errs1 = q_routing(lengths1, Q=Q0, seed=1)
assert errs1[-1] < 1e-3

def greedy_route(Q):
    v, route = s, [s]
    while v != t:
        k = out_arcs[v][int(np.argmax(Q[out_arcs[v]]))]
        v = ARCS[k][1]; route.append(v)
    return route

print("route before:", greedy_route(Q0), "  after:", greedy_route(Q1))

plt.semilogy(np.arange(len(errs0)) * 25, np.array(errs0) + 1e-12, label="original network")
plt.semilogy(np.arange(len(errs1)) * 25 + len(errs0) * 25, np.array(errs1) + 1e-12, label="after interdiction")
plt.axvline(len(errs0) * 25, color="k", ls=":")
plt.xlabel("episode"); plt.ylabel("max |V + dist|"); plt.legend(); plt.show()

## Where to go from here

* **Stochastic travel times:** make lengths random. Dijkstra on expected values is still optimal for expected cost, but Q-learning needs decaying $\alpha$. Risk-averse objectives (CVaR) are where RL starts to differ from classical OR.
* **Learning the adversary:** let the interdictor learn too (two-player Q-learning or policy gradient), and compare the equilibrium with the MILP solution from notebook 12.
* **Scaling up:** replace the table with a graph neural network, $Q_\theta(v,\text{arc})$. This is the bridge to "learning to optimize" research.

## Exercises

1. Add Gaussian noise to every arc length at each traversal. Compare constant $\alpha$ and $\alpha_n=1/n(s,a)$.
2. Count the episodes needed to recover after interdiction, as a function of $\varepsilon$. Why does too little exploration hurt here?